# Controlador multisemilla para Senior Exp6

Esta notebook corre la workflow base `z635_WorkFlow_01_senior_exp6.ipynb` una vez por cada semilla.
En cada corrida se setean:
- `PARAM$semilla_primigenia` con el valor de la lista `semillas`
- `PARAM$experimento` con un entero incremental en 1
- `PARAM$CN$ratio`
- `PARAM$CN$desvios`
- `PARAM$boruta$aplicacion_boruta`

In [1]:
CONFIG <- list(
  semillas = c(109849, 109859, 109873), #109843, 109847 
  nro_experimento_inicial = 9501L,
  experimento_state_file = "./exp6_multisemilla/ultimo_experimento.txt",
  ratio_canaritos = 0.2,
  desvios_canaritos = 2,
  aplicacion_boruta_flag = FALSE,
  notebook_base = "z635_WorkFlow_01_senior_exp6.ipynb",
  output_dir = "./exp6_multisemilla"
)

In [2]:
leer_notebook <- function(path) {
  nb <- jsonlite::fromJSON(path, simplifyVector = FALSE)
  nb$cells
}

requerir_parametro <- function(nombre, entorno = .GlobalEnv) {
  if (!exists("PARAM", mode = "list", where = entorno, inherits = TRUE)) {
    stop("Falta establecer PARAM en la notebook controladora. Debes definirlo antes de correr la base.", call. = FALSE)
  }
  if (is.null(get("PARAM", envir = entorno, inherits = TRUE)[[nombre]])) {
    stop(paste0("Falta establecer PARAM$", nombre, " en la notebook controladora."), call. = FALSE)
  }
  invisible(get("PARAM", envir = entorno, inherits = TRUE)[[nombre]])
}

requerir_subparametro <- function(grupo, nombre, entorno = .GlobalEnv) {
  if (!exists("PARAM", mode = "list", where = entorno, inherits = TRUE)) {
    stop("Falta establecer PARAM en la notebook controladora. Debe definirse antes de correr la base.", call. = FALSE)
  }
  param_obj <- get("PARAM", envir = entorno, inherits = TRUE)
  if (is.null(param_obj[[grupo]]) || is.null(param_obj[[grupo]][[nombre]])) {
    stop(paste0("Falta establecer PARAM$", grupo, "$", nombre, " en la notebook controladora."), call. = FALSE)
  }
  invisible(param_obj[[grupo]][[nombre]])
}

verificar_parametros <- function(entorno = .GlobalEnv) {
  requerir_parametro("semilla_primigenia", entorno = entorno)
  requerir_parametro("experimento", entorno = entorno)
  requerir_subparametro("CN", "ratio", entorno = entorno)
  requerir_subparametro("CN", "desvios", entorno = entorno)
  requerir_subparametro("boruta", "aplicacion_boruta", entorno = entorno)
  TRUE
}

reemplazar_si_existe <- function(src, patron, reemplazo) {
  if (grepl(patron, src, perl = TRUE)) {
    gsub(patron, reemplazo, src, perl = TRUE)
  } else {
    src
  }
}

ejecutar_corrida <- function(semilla, experimento, ratio, desvios, boruta_flag, base_path, work_dir) {
  dir.create(work_dir, recursive = TRUE, showWarnings = FALSE)
  entorno_corrida <- new.env(parent = globalenv())
  assign("PARAM", list(
    semilla_primigenia = semilla,
    experimento = experimento,
    CN = list(ratio = ratio, desvios = desvios),
    boruta = list(aplicacion_boruta = boruta_flag)
  ), envir = entorno_corrida)

  verificar_parametros(entorno = entorno_corrida)

  inicio_corrida <- Sys.time()
  cat("\n[Inicio corrida] semilla=", semilla, "| experimento=", experimento, "| ratio=", ratio, "| desvios=", desvios, "| boruta=", boruta_flag, "| horario_inicio=", format(inicio_corrida, "%Y-%m-%d %H:%M:%S"), "\n")
  flush.console()

  cells <- leer_notebook(base_path)
  for (i in seq_along(cells)) {
    cell <- cells[[i]]
    if (is.null(cell$cell_type) || !identical(cell$cell_type, "code")) next

    src <- paste(unlist(cell$source), collapse = "")

    src <- reemplazar_si_existe(src, "PARAM\\$semilla_primigenia\\s*<-\\s*.*", paste0("PARAM$semilla_primigenia <- ", semilla))
    src <- reemplazar_si_existe(src, "PARAM\\$experimento\\s*<-\\s*.*", paste0("PARAM$experimento <- ", experimento))
    src <- reemplazar_si_existe(src, "PARAM\\$CN\\$ratio\\s*<-\\s*.*", paste0("PARAM$CN$ratio <- ", ratio))
    src <- reemplazar_si_existe(src, "PARAM\\$CN\\$desvios\\s*<-\\s*.*", paste0("PARAM$CN$desvios <- ", desvios))
    src <- reemplazar_si_existe(src, "PARAM\\$boruta\\$aplicacion_boruta\\s*<-\\s*.*", paste0("PARAM$boruta$aplicacion_boruta <- ", if (boruta_flag) "TRUE" else "FALSE"))

    if (i == 1L) {
      src <- paste0("setwd(", shQuote(work_dir), ")\n", src)
    }

    cat("\n[Celda ", i, " / ", length(cells), "] ejecutando...\n")
    flush.console()
    eval(parse(text = src), envir = entorno_corrida)
    cat("[Celda ", i, " / ", length(cells), "] OK\n")
    flush.console()
  }

  fin_corrida <- Sys.time()
  cat("\n[Fin corrida] semilla=", semilla, "| experimento=", experimento, "| horario_fin=", format(fin_corrida, "%Y-%m-%d %H:%M:%S"), "| duracion_segundos=", round(as.numeric(difftime(fin_corrida, inicio_corrida, units = "secs")), 1), "\n")
  flush.console()

  invisible(NULL)
}



In [ ]:
base_path <- normalizePath(file.path(getwd(), CONFIG$notebook_base), winslash = "/")
if (!file.exists(base_path)) stop("No existe la notebook base: ", base_path)

leer_ultimo_experimento <- function(path) {
  if (!file.exists(path)) {
    return(NULL)
  }
  valor <- suppressWarnings(as.integer(readLines(path, warn = FALSE)[1]))
  if (is.na(valor)) {
    stop("El archivo de estado del experimento existe pero no tiene un número válido: ", path, call. = FALSE)
  }
  valor
}

guardar_ultimo_experimento <- function(path, valor) {
  dir.create(dirname(path), recursive = TRUE, showWarnings = FALSE)
  writeLines(as.character(valor), con = path, useBytes = TRUE)
}

ultimo_experimento <- leer_ultimo_experimento(CONFIG$experimento_state_file)
if (is.null(ultimo_experimento)) {
  ultimo_experimento <- as.integer(CONFIG$nro_experimento_inicial) - 1L
}

for (i in seq_along(CONFIG$semillas)) {
  ultimo_experimento <- ultimo_experimento + 1L
  semilla <- CONFIG$semillas[i]
  experimento <- ultimo_experimento
  run_dir <- file.path(CONFIG$output_dir, paste0("exp_", experimento, "_s_", semilla))

  cat("\n=== Corrida", i, "de", length(CONFIG$semillas), "===\n")
  cat("semilla:", semilla, "| experimento:", experimento, "| ratio:", CONFIG$ratio_canaritos, "| desvios:", CONFIG$desvios_canaritos, "| boruta:", CONFIG$aplicacion_boruta_flag, "\n")

  PARAM <- NULL

  ejecutar_corrida(
    semilla = semilla,
    experimento = experimento,
    ratio = CONFIG$ratio_canaritos,
    desvios = CONFIG$desvios_canaritos,
    boruta_flag = CONFIG$aplicacion_boruta_flag,
    base_path = base_path,
    work_dir = run_dir
  )

  guardar_ultimo_experimento(CONFIG$experimento_state_file, experimento)
}



=== Corrida 1 de 3 ===
semilla: 109849 | experimento: 9501 | ratio: 0.2 | desvios: 2 | boruta: FALSE 

[Inicio corrida] semilla= 109849 | experimento= 9501 | ratio= 0.2 | desvios= 2 | boruta= FALSE | horario_inicio= 2026-10-31 14:05:47 

[Celda  8  /  125 ] ejecutando...
[Celda  8  /  125 ] OK

[Celda  9  /  125 ] ejecutando...
[Celda  9  /  125 ] OK

[Celda  10  /  125 ] ejecutando...


Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: R.utils

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Attaching package: ‘R.utils’


The following object is masked from ‘package:utils’:

    timestamp


The following objects are masked from ‘package:base’:

    cat, commandArgs, getOption, isOpen, nullfile, parse, u

[Celda  10  /  125 ] OK

[Celda  12  /  125 ] ejecutando...
[Celda  12  /  125 ] OK

[Celda  14  /  125 ] ejecutando...
[Celda  14  /  125 ] OK

[Celda  17  /  125 ] ejecutando...
[Celda  17  /  125 ] OK

[Celda  20  /  125 ] ejecutando...


Loading required package: mice


Attaching package: ‘mice’


The following object is masked from ‘package:stats’:

    filter


The following objects are masked from ‘package:base’:

    cbind, rbind




[Celda  20  /  125 ] OK

[Celda  21  /  125 ] ejecutando...
[Celda  21  /  125 ] OK

[Celda  22  /  125 ] ejecutando...
[Celda  22  /  125 ] OK

[Celda  23  /  125 ] ejecutando...
[Celda  23  /  125 ] OK

[Celda  24  /  125 ] ejecutando...
[Celda  24  /  125 ] OK

[Celda  25  /  125 ] ejecutando...
[Celda  25  /  125 ] OK

[Celda  26  /  125 ] ejecutando...
inicio Corregir_Rotas()
fin Corregir_rotas()
[Celda  26  /  125 ] OK

[Celda  28  /  125 ] ejecutando...
[Celda  28  /  125 ] OK

[Celda  29  /  125 ] ejecutando...
[Celda  29  /  125 ] OK

[Celda  30  /  125 ] ejecutando...
[Celda  30  /  125 ] OK

[Celda  31  /  125 ] ejecutando...
[Celda  31  /  125 ] OK

[Celda  32  /  125 ] ejecutando...
[Celda  32  /  125 ] OK

[Celda  33  /  125 ] ejecutando...
[Celda  33  /  125 ] OK

[Celda  34  /  125 ] ejecutando...
[Celda  34  /  125 ] OK

[Celda  35  /  125 ] ejecutando...
[Celda  35  /  125 ] OK

[Celda  36  /  125 ] ejecutando...
[Celda  36  /  125 ] OK

[Celda  37  /  125 ] ejecutand

...

[Celda  85  /  125 ] OK

[Celda  87  /  125 ] ejecutando...
[Celda  87  /  125 ] OK

[Celda  88  /  125 ] ejecutando...
[Celda  88  /  125 ] OK

[Celda  90  /  125 ] ejecutando...
[Celda  90  /  125 ] OK

[Celda  92  /  125 ] ejecutando...
[Celda  92  /  125 ] OK

[Celda  94  /  125 ] ejecutando...


Computing y column(s) for design. Not provided.

Sat Oct 31 14:16:41 2026  234, 73, 0.102423855450746, 0.396672157139473, 0.0728017310187789 AUC 0.931729127401473

Sat Oct 31 14:16:51 2026  44, 1457, 0.179741687948345, 0.634894958513475, 102.935661935966 AUC 0.934271430859512

Sat Oct 31 14:17:05 2026  80, 121, 0.265965978005075, 0.301477679550847, 0.0386976039059872 AUC 0.911121432876162

Sat Oct 31 14:18:54 2026  452, 438, 0.0682481725927501, 0.441818114035569, 11.6609642257469 AUC 0.935944735553141

Sat Oct 31 14:19:02 2026  92, 15, 0.292069573377365, 0.698158891441894, 31.688618298176 AUC 0.931498575293493

Sat Oct 31 14:19:05 2026  19, 9, 0.0313064204402014, 0.150571968750206, 0.0276776649163583 AUC 0.903704403366998

Sat Oct 31 14:19:09 2026  28, 39, 0.0202385974332323, 0.77652264818589, 6.4443837888462 AUC 0.914820114823252

Sat Oct 31 14:19:21 2026  58, 24, 0.922010108930137, 0.540394879338479, 0.00361472921674788 AUC 0.807187282160482

Sat Oct 31 14:19:24 2026  4, 1057, 0.1146

[Celda  94  /  125 ] OK

[Celda  96  /  125 ] ejecutando...
$num_iterations
[1] 9.998666

$num_leaves
[1] 10.9419

$learning_rate
[1] -6.674257

$feature_fraction
[1] 0.3578535

$min_sum_hessian_in_leaf
[1] -1.369102

   num_iterations num_leaves learning_rate feature_fraction
            <num>      <num>         <num>            <num>
1:       9.998666    10.9419     -6.674257        0.3578535
   min_sum_hessian_in_leaf         y
                     <num>     <num>
1:               -1.369102 0.9388217
[Celda  96  /  125 ] OK

[Celda  97  /  125 ] ejecutando...
[Celda  97  /  125 ] OK

[Celda  101  /  125 ] ejecutando...
[Celda  101  /  125 ] OK

[Celda  103  /  125 ] ejecutando...
[Celda  103  /  125 ] OK

[Celda  104  /  125 ] ejecutando...
[Celda  104  /  125 ] OK

[Celda  106  /  125 ] ejecutando...
[Celda  106  /  125 ] OK

[Celda  107  /  125 ] ejecutando...


Loading required package: primes



[Celda  107  /  125 ] OK

[Celda  108  /  125 ] ejecutando...
453569 993253 507499 599143 325751 924283 767863 380503 106591 107057[Celda  108  /  125 ] OK

[Celda  109  /  125 ] ejecutando...
[Celda  109  /  125 ] OK

[Celda  110  /  125 ] ejecutando...
